Подключение необходимых библиотек:
pandas — основная библиотека для работы с таблицами (CSV-файлами)
re — библиотека для работы с регулярными выражениями (поиск текстовых аномалий)
os — стандартная библиотека для работы с путями и файловой системой


In [ ]:
import pandas as pd
import re
import os



=== БЛОК 1: НАСТРОЙКА ПУТЕЙ ===
Указываем рабочую директорию и полные пути к исходному и результирующему файлам


In [ ]:
base_dir = r"C:\Users\user\Desktop\Работа\Почти финал"
input_file = os.path.join(base_dir, "literaguru_final_clean_1343.csv")
output_file = os.path.join(base_dir, "literaguru_strict_clean.csv")



=== БЛОК 2: ЗАГРУЗКА ДАННЫХ ===
Читаем CSV-файл. Используем разделитель ';' и кодировку UTF-8 для корректного отображения кириллицы


In [ ]:
df = pd.read_csv(input_file, sep=';', encoding='utf-8')



=== БЛОК 3: ПОДГОТОВКА ШАБЛОНОВ ФИЛЬТРАЦИИ ===
homoglyph_pattern: ищет смешивание русских и английских букв внутри одного слова (защита от "склейки" и обхода антиплагиата)


In [ ]:
homoglyph_pattern = re.compile(r'[а-яёА-ЯЁ][a-zA-Z]|[a-zA-Z][а-яёА-ЯЁ]')
# broken_parsing_pattern: ищет ошибки парсинга, когда предложение заканчивается знаком препинания, но следующее начинается с маленькой буквы
broken_parsing_pattern = re.compile(r'(\.|\!|\?)\s+[а-яёa-z]')

# === БЛОК 4: ЛОГИКА ПРОВЕРКИ ТЕКСТА НА БРАК ===
def is_defective(text):
    # Если в ячейке не текст (пустота), помечаем как брак
    if not isinstance(text, str):
        return True

    text_stripped = text.strip()
    # Если после удаления пробелов текст пуст — это брак
    if not text_stripped:
        return True

    # Если текст начинается с маленькой буквы — это признак "обрубка" или ошибки парсинга
    if text_stripped[0].islower():
        return True

    # Проверка на ошибки пунктуации (маленькая буква после точки)
    if broken_parsing_pattern.search(text_stripped):
        return True

    # Проверка на наличие хомоглифов (смешанная раскладка букв в словах)
    if homoglyph_pattern.search(text_stripped):
        return True

    # Если ни одна проверка не сработала — текст качественный
    return False



=== БЛОК 5: ПРИМЕНЕНИЕ ФИЛЬТРАЦИИ ===
Создаем "маску" (список меток), где оставляем только те тексты, которые НЕ являются дефектными


In [ ]:
mask = ~df['Текст'].apply(is_defective)
df_cleaned = df[mask]



=== БЛОК 6: СОХРАНЕНИЕ РЕЗУЛЬТАТА ===
Сохраняем очищенные данные. Используем кодировку 'utf-8-sig'.
Это крайне важно для того, чтобы Excel открывал файл сразу в правильной кодировке и с кириллицей.


In [ ]:
df_cleaned.to_csv(output_file, sep=';', index=False, encoding='utf-8-sig')



=== БЛОК 7: ОТЧЕТ ДЛЯ ПОЛЬЗОВАТЕЛЯ ===
Вывод краткой статистики по результатам работы скрипта


In [ ]:
print(f"Исходное количество текстов: {len(df)}")
print(f"Осталось текстов: {len(df_cleaned)}")
print(f"Удалено текстов (хомоглифы + битый парсинг): {len(df) - len(df_cleaned)}")
